## Workflow for Neosurf-on-Neosurf MaSIF search

In [25]:
import os
import pandas as pd

repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]
os.chdir(repo_root)

# ----- Directories ------
data_dir = os.path.join(repo_root, 'data')

# ----- Input ------
# .csv about the seed complexes
seed_list_csv = os.path.join(data_dir, 'nico_targets.csv')

# Directory to store input .pdb and .sdf
input_dir = os.path.join(data_dir, 'input')
os.makedirs(input_dir, exist_ok=True)
input_manifest = os.path.join(input_dir, 'input_manifest.csv')

# ----- Output ------
# Directory to write preprocessing files
preprocess_dir = os.path.join(data_dir, 'preprocess')

# Directory to write masif-search output
masif_search_out_dir = os.path.join(data_dir, 'masif_search')
os.makedirs(masif_search_out_dir, exist_ok=True)


___
### Step 1 - preprocess all targets
1. Preprocess targets with ligands in nico_targets.csv
2. Preprocess VHL and CRBN with ligands

In [3]:
df_seed = pd.read_csv(seed_list_csv)

df_seed.head()

,uniprot_id,gene_name,recommendedName,pdb_id,protein_chain,ligand_chain,ligand_code,ligand_name,smiles,percent_intracellular,formula,mw,qed,num_carbon,num_N_O,uniprot_id_count
0,P35222,CTNNB1,Catenin beta-1,6M90,C,A,J91,2-(2-fluorophenoxy)-3-{[2-oxo-6-(trifluorometh...,c1ccc(c(c1)Oc2c(cccc2NC(=O)C3=CC=C(NC3=O)C(F)(...,1.0,C20H12F4N2O5,436.068234,0.517802,20.0,7.0,1.0
1,P35222,CTNNB1,Catenin beta-1,6M91,C,A,J97,"3-({4-[(2,6-dichlorophenyl)sulfanyl]-2-oxo-6-(...",c1cc(cc(c1)NC(=O)C2=C(C=C(NC2=O)C(F)(F)F)Sc3c(...,1.0,C20H11Cl2F3N2O4S,501.976868,0.408829,20.0,6.0,1.0
2,P35222,CTNNB1,Catenin beta-1,6M92,C,A,J8V,"3-{[2-oxo-4-phenoxy-6-(trifluoromethyl)-1,2-di...",c1ccc(cc1)OC2=C(C(=O)NC(=C2)C(F)(F)F)C(=O)Nc3c...,1.0,C20H13F3N2O5,418.077656,0.577285,20.0,7.0,1.0
3,P35222,CTNNB1,Catenin beta-1,6M93,C,A,J8Y,2-oxo-N-[3-(1H-tetrazol-5-yl)phenyl]-6-(triflu...,c1cc(cc(c1)NC(=O)C2=CC=C(NC2=O)C(F)(F)F)c3[nH]...,1.0,C14H9F3N6O2,350.073908,0.666230,14.0,8.0,1.0
4,P35222,CTNNB1,Catenin beta-1,7AFW,A,A,R9Q,"3-[(2~{R})-4-methyl-5-oxidanylidene-2,3-dihydr...",CN1C[C@H](Oc2ccccc2C1=O)c3cccc(c3)C#N,1.0,C17H14N2O2,278.105528,0.805631,17.0,4.0,1.0


In [9]:
# ----- Helper functions for preparing preprocessing inputs ------
#
# prepare_input_structures() orchestrates one row of df_seed (nico_targets.csv) into
# MaSIF-ready files under input_dir. Per complex the workflow is:
#
#   1. Download the full PDB from RCSB and the ligand SDF (models.rcsb.org) into
#      input_dir / a temp directory.
#   2. Trim the structure to the target protein (standard amino acids on protein_chain)
#      plus a single ligand residue (ligand_code on ligand_chain). If protein_chain
#      and ligand_chain are the same, both are kept on that chain.
#   3. Build a target name {pdb_id}_{chains}, where chains = protein_chain + ligand_chain
#      deduplicated (e.g. C+A -> CA, A+A -> A).
#   4. EvoEF2 RepairStructure on a protein-only PDB (ligand stripped); repaired
#      coordinates are written to a temp file as structure_Repair.pdb.
#   5. Merge the original ligand HETATM/ATOM records from the pre-repair trimmed complex
#      onto the repaired protein and save the final PDB to input_dir.
#   6. Return a dict for df_preprocess: pdb_path, target, ligand ({code}_{chain}),
#      ligand_path ({pdb_id}_{protein_chain}_{ligand_code}.sdf).
#
# Files are always rebuilt (no skip-if-exists). The driver cell loops df_seed and
# calls prepare_input_structures(row, input_dir, repo_root/EvoEF2/EvoEF2).

import shutil
import subprocess
import tempfile
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import urlopen

from Bio.PDB import PDBIO, PDBParser, Select

# chains suffix: protein_chain + ligand_chain, deduplicated (C+A -> CA, A+A -> A)
STANDARD_AA = {
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL",
    "MSE",
}

evoef2_bin = os.path.join(repo_root, "EvoEF2", "EvoEF2")

def chain_suffix(protein_chain, ligand_chain):
    if protein_chain == ligand_chain:
        return protein_chain
    return protein_chain + ligand_chain


def _is_standard_protein_residue(residue):
    return residue.id[0] == " " and residue.get_resname().strip() in STANDARD_AA


def _is_ligand_residue(residue, ligand_code):
    return residue.get_resname().strip() == ligand_code.strip()


class _ComplexSelect(Select):
    def __init__(self, protein_chain, ligand_chain, ligand_code):
        self.protein_chain = protein_chain
        self.ligand_chain = ligand_chain
        self.ligand_code = ligand_code
        self._chains = {protein_chain, ligand_chain}

    def accept_chain(self, chain):
        return chain.id in self._chains

    def accept_residue(self, residue):
        chain_id = residue.parent.id
        if chain_id == self.protein_chain:
            if self.protein_chain == self.ligand_chain:
                return _is_standard_protein_residue(residue) or _is_ligand_residue(
                    residue, self.ligand_code
                )
            return _is_standard_protein_residue(residue)
        if chain_id == self.ligand_chain:
            return _is_ligand_residue(residue, self.ligand_code)
        return False


class _ProteinOnlySelect(Select):
    def __init__(self, protein_chain):
        self.protein_chain = protein_chain

    def accept_chain(self, chain):
        return chain.id == self.protein_chain

    def accept_residue(self, residue):
        return _is_standard_protein_residue(residue)


def _download_url(url, dest_path):
    dest_path = Path(dest_path)
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urlopen(url) as response:
            dest_path.write_bytes(response.read())
    except HTTPError as exc:
        raise RuntimeError(f"Download failed ({exc.code}): {url}") from exc


def download_pdb(pdb_id, dest_path):
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    _download_url(url, dest_path)


def download_ligand_sdf(pdb_id, ligand_chain, dest_path):
    url = (
        f"https://models.rcsb.org/v1/{pdb_id}/ligand"
        f"?label_asym_id={ligand_chain}&encoding=sdf"
    )
    _download_url(url, dest_path)


def _extract_ligand_pdb_lines(pdb_path, ligand_chain, ligand_code):
    lines = []
    with open(pdb_path) as handle:
        for line in handle:
            if not (line.startswith("HETATM") or line.startswith("ATOM")):
                continue
            if line[21] != ligand_chain:
                continue
            if line[17:20].strip() != ligand_code.strip():
                continue
            lines.append(line)
    if not lines:
        raise ValueError(
            f"No ligand records for resname={ligand_code} chain={ligand_chain} in {pdb_path}"
        )
    return lines


def _merge_repaired_protein_with_ligand(repaired_pdb, trimmed_pdb, ligand_chain, ligand_code, output_pdb):
    ligand_lines = _extract_ligand_pdb_lines(trimmed_pdb, ligand_chain, ligand_code)
    out_lines = []
    with open(repaired_pdb) as handle:
        for line in handle:
            if line.startswith("END"):
                break
            if line.strip():
                out_lines.append(line)
    out_lines.extend(ligand_lines)
    out_lines.append("END\n")
    Path(output_pdb).write_text("".join(out_lines))


def evoef2_repair_structure(input_pdb, output_pdb, evoef2_bin):
    """Run EvoEF2 RepairStructure; intermediate {stem}_Repair.pdb lives in a temp directory."""
    evoef2_bin = str(evoef2_bin)
    if not os.path.isfile(evoef2_bin):
        raise FileNotFoundError(
            f"EvoEF2 binary not found at {evoef2_bin}. Build with: cd EvoEF2 && ./build.sh"
        )
    output_pdb = Path(output_pdb)
    output_pdb.parent.mkdir(parents=True, exist_ok=True)

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        stem = "structure"
        work_pdb = tmp / f"{stem}.pdb"
        shutil.copy2(input_pdb, work_pdb)
        subprocess.run(
            [
                evoef2_bin,
                "--command=RepairStructure",
                f"--pdb={stem}.pdb"
            ],
            cwd=tmp,
            check=True,
        )
        repaired = tmp / f"{stem}_Repair.pdb"
        if not repaired.is_file():
            raise FileNotFoundError(f"EvoEF2 did not create {repaired}")
        shutil.copy2(repaired, output_pdb)


def prepare_input_structures(pdb_id, protein_chain, ligand_chain, ligand_code, input_dir, evoef2_bin):
    """
    Download PDB/SDF, trim to protein+ligand, EvoEF2-repair protein, merge ligand, write to input_dir.

    chains suffix: protein_chain + ligand_chain (deduplicated if equal).
    """
    input_dir = Path(input_dir)

    chains = chain_suffix(protein_chain, ligand_chain)
    target = f"{pdb_id}_{chains}"
    ligand_name = f"{ligand_code}_{ligand_chain}"
    pdb_path = input_dir / f"{target}.pdb"
    ligand_path = input_dir / f"{pdb_id}_{protein_chain}_{ligand_code}.sdf"

    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        full_pdb = tmp / f"{pdb_id}.pdb"
        download_pdb(pdb_id, full_pdb)
        download_ligand_sdf(pdb_id, ligand_chain, ligand_path)

        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(pdb_id, str(full_pdb))

        trimmed_pdb = tmp / "trimmed.pdb"
        pdb_io = PDBIO()
        pdb_io.set_structure(structure)
        pdb_io.save(str(trimmed_pdb), _ComplexSelect(protein_chain, ligand_chain, ligand_code))
        _extract_ligand_pdb_lines(trimmed_pdb, ligand_chain, ligand_code)

        protein_only_pdb = tmp / "protein_only.pdb"
        pdb_io.set_structure(structure)
        pdb_io.save(str(protein_only_pdb), _ProteinOnlySelect(protein_chain))

        repaired_protein_pdb = tmp / "repaired_protein.pdb"
        evoef2_repair_structure(
            protein_only_pdb, repaired_protein_pdb, evoef2_bin
        )
        _merge_repaired_protein_with_ligand(
            repaired_protein_pdb, trimmed_pdb, ligand_chain, ligand_code, pdb_path
        )

    return {
        "pdb_path": str(pdb_path.resolve()),
        "target": target,
        "ligand": ligand_name,
        "ligand_path": str(ligand_path.resolve()),
    }



In [ ]:
# Iterate over all rows in df_seed

rows = []
for _, row in df_seed.iterrows():
    print(f"Preparing {row['pdb_id']} ({row['protein_chain']} + {row['ligand_chain']})...")
    pdb_id = row["pdb_id"]
    protein_chain = row["protein_chain"]
    ligand_chain = row["ligand_chain"]
    ligand_code = row["ligand_code"]
    rows.append(prepare_input_structures(pdb_id, protein_chain, ligand_chain, ligand_code, input_dir, evoef2_bin))

df_preprocess = pd.DataFrame(rows)
df_preprocess

Preparing 6M90 (C + A)...
############################################################################################
                                    EvoEF2                                                  
  A framework for macromolecular modeling, e.g.,protein design, protein side-chain packing, 
protein structure energy minimization, add and optimize hydrogen bonds, build mutant model, 
calculate protein folding stability, calculate protein-protein binding free energy, etc     


  Copyright (c) Xiaoqiang Huang (xiaoqiah@umich.edu; tommyhuangthu@foxmail.com)
  Dept. of Computational Medicine & Bioinformatics
  Medical School
  University of Michigan
############################################################################################
command RepairStructure works
EvoEF Repairing PDB: optimization cycle 1 ... 
We optimize side chain of residue C31L
We optimize side chain of residue C32D
We optimize side chain of residue C35I
We will flip residue C36H to optimize hbond
We 

,pdb_path,target,ligand,ligand_path
0,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,6M90_CA,J91_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
1,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,6M91_CA,J97_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
2,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,6M92_CA,J8V_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
3,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,6M93_CA,J8Y_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
4,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,7AFW_A,R9Q_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
5,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,7UWO_AB,WHL_B,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
6,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,7ZRB_A,JKI_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
7,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,8EI9_C,WHL_C,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
8,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,8EIA_AC,WHL_C,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
9,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,8EIB_AC,WHL_C,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...


In [10]:
# Add VHL structure as well:
pdb_id = "8VLB"
protein_chain = "A"
ligand_chain = "A"
ligand_code = "3JF"


rows.append(prepare_input_structures(pdb_id, protein_chain, ligand_chain, ligand_code, input_dir, evoef2_bin))

df_preprocess = pd.DataFrame(rows)
df_preprocess

############################################################################################
                                    EvoEF2                                                  
  A framework for macromolecular modeling, e.g.,protein design, protein side-chain packing, 
protein structure energy minimization, add and optimize hydrogen bonds, build mutant model, 
calculate protein folding stability, calculate protein-protein binding free energy, etc     


  Copyright (c) Xiaoqiang Huang (xiaoqiah@umich.edu; tommyhuangthu@foxmail.com)
  Dept. of Computational Medicine & Bioinformatics
  Medical School
  University of Michigan
############################################################################################
command RepairStructure works
EvoEF Repairing PDB: optimization cycle 1 ... 
We optimize side chain of residue A61P
We optimize side chain of residue A62V
We optimize side chain of residue A63L
We optimize side chain of residue A64R
We will rotate hydroxyl group of r

,pdb_path,target,ligand,ligand_path
0,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,6M90_CA,J91_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
1,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,6M91_CA,J97_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
2,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,6M92_CA,J8V_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
3,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,6M93_CA,J8Y_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
4,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,7AFW_A,R9Q_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
5,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,7UWO_AB,WHL_B,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
6,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,7ZRB_A,JKI_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
7,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,8EI9_C,WHL_C,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
8,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,8EIA_AC,WHL_C,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
9,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,8EIB_AC,WHL_C,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...


In [11]:
# Add CRBN-pomalidomide structure as well:
pdb_id = "6H0F"
protein_chain = "B"
ligand_chain = "B"
ligand_code = "Y70"

rows.append(prepare_input_structures(pdb_id, protein_chain, ligand_chain, ligand_code, input_dir, evoef2_bin))

df_preprocess = pd.DataFrame(rows)
df_preprocess

############################################################################################
                                    EvoEF2                                                  
  A framework for macromolecular modeling, e.g.,protein design, protein side-chain packing, 
protein structure energy minimization, add and optimize hydrogen bonds, build mutant model, 
calculate protein folding stability, calculate protein-protein binding free energy, etc     


  Copyright (c) Xiaoqiang Huang (xiaoqiah@umich.edu; tommyhuangthu@foxmail.com)
  Dept. of Computational Medicine & Bioinformatics
  Medical School
  University of Michigan
############################################################################################
command RepairStructure works
EvoEF Repairing PDB: optimization cycle 1 ... 
We optimize side chain of residue B70R
We will rotate hydroxyl group of residue B71T to optimize hbond
We optimize side chain of residue B71T
We optimize side chain of residue B72L
We will f

,pdb_path,target,ligand,ligand_path
0,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,6M90_CA,J91_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
1,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,6M91_CA,J97_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
2,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,6M92_CA,J8V_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
3,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,6M93_CA,J8Y_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
4,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,7AFW_A,R9Q_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
5,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,7UWO_AB,WHL_B,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
6,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,7ZRB_A,JKI_A,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
7,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,8EI9_C,WHL_C,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
8,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,8EIA_AC,WHL_C,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...
9,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...,8EIB_AC,WHL_C,/scratch/ymeng/NNeosurf/masif-neosurf/data/inp...


In [24]:
# Write df_preprocess to input_manifest.csv, then run preprocess_array.sh
df_preprocess.to_csv(input_manifest, index=False)
n_rows = len(df_preprocess)
manifest_abs = os.path.abspath(input_manifest)
# Use {var} for Python values in ! commands — ${var} becomes a literal '$' + path in Jupyter
!sbatch --array=1-{n_rows} scripts/slurm/preprocess_array.sh {manifest_abs}

sbatch: [ESTIMATION] The estimated cost of this job is CHF 0.05
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 1.25        │ 0.1         │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 1.45        │ 0.1         │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)
Submitted batch job 54908372


___
### Step 2 - Run masif_search around ligand residues
1. Query with VHL structure as target, search for seed patches around the seed ligands

In [26]:
import numpy as np
from Bio.PDB import PDBParser

QUERY_TARGET = "8VLB_A"
query_out_dir = os.path.join(masif_search_out_dir, QUERY_TARGET)
os.makedirs(query_out_dir, exist_ok=True)

search_manifest = os.path.join(query_out_dir, "search_manifest.csv")


def parse_ligand_name(ligand):
    code, chain = ligand.rsplit("_", 1)
    return code.strip(), chain.strip()


def protein_chain_from_target(target, ligand_chain):
    _, chains = target.split("_", 1)
    if chains == ligand_chain:
        return ligand_chain
    if chains.endswith(ligand_chain):
        protein = chains[: -len(ligand_chain)]
        if protein:
            return protein
    raise ValueError(f"Cannot infer protein chain from {target=} {ligand_chain=}")


def ligand_anchor_from_row(row):
    """Return (ligand_chain, ligand_resid) from pdb_path and ligand column."""
    ligand_code, ligand_chain = parse_ligand_name(row["ligand"])
    structure = PDBParser(QUIET=True).get_structure(row["target"], row["pdb_path"])
    residues = [
        r
        for r in structure[0][ligand_chain].get_residues()
        if r.get_resname().strip() == ligand_code
    ]
    if not residues:
        raise ValueError(f"No {ligand_code} on chain {ligand_chain} in {row['pdb_path']}")
    if len(residues) == 1:
        return ligand_chain, int(residues[0].id[1])

    protein_chain = protein_chain_from_target(row["target"], ligand_chain)
    ref = np.mean(
        [r["CA"].get_coord() for r in structure[0][protein_chain].get_residues() if r.id[0] == " " and "CA" in r],
        axis=0,
    )
    best = min(
        residues,
        key=lambda r: np.linalg.norm(
            np.mean([a.get_coord() for a in r.get_atoms() if a.element != "H"], axis=0) - ref
        ),
    )
    return ligand_chain, int(best.id[1])


query_row = df_preprocess.loc[df_preprocess["target"] == QUERY_TARGET].iloc[0]
target_chain, target_resid = ligand_anchor_from_row(query_row)

search_rows = []
for _, seed_row in df_preprocess[df_preprocess["target"] != QUERY_TARGET].iterrows():
    seed_chain, seed_resid = ligand_anchor_from_row(seed_row)
    out_dir = query_out_dir
    search_rows.append(
        {
            "query_target": QUERY_TARGET,
            "target_chain": target_chain,
            "target_resid": target_resid,
            "seed_target": seed_row["target"],
            "seed_chain": seed_chain,
            "seed_resid": seed_resid,
            "out_dir": out_dir,
        }
    )

df_search = pd.DataFrame(search_rows)
df_search.to_csv(search_manifest, index=False)
print(f"Query {QUERY_TARGET}: chain={target_chain} resid={target_resid}")
print(f"Wrote {len(df_search)} search jobs to {search_manifest}")
df_search.head()


Query 8VLB_A: chain=A resid=301
Wrote 27 search jobs to /scratch/ymeng/NNeosurf/masif-neosurf/data/masif_search/8VLB_A/search_manifest.csv


,query_target,target_chain,target_resid,seed_target,seed_chain,seed_resid,out_dir
0,8VLB_A,A,301,6M90_CA,A,601,/scratch/ymeng/NNeosurf/masif-neosurf/data/mas...
1,8VLB_A,A,301,6M91_CA,A,601,/scratch/ymeng/NNeosurf/masif-neosurf/data/mas...
2,8VLB_A,A,301,6M92_CA,A,601,/scratch/ymeng/NNeosurf/masif-neosurf/data/mas...
3,8VLB_A,A,301,6M93_CA,A,601,/scratch/ymeng/NNeosurf/masif-neosurf/data/mas...
4,8VLB_A,A,301,7AFW_A,A,401,/scratch/ymeng/NNeosurf/masif-neosurf/data/mas...


In [ ]:
n_search = len(df_search)
search_manifest_abs = os.path.abspath(search_manifest)
!sbatch --array=1-{n_search} scripts/slurm/search_array.sh {search_manifest_abs}

sbatch: [ESTIMATION] The estimated cost of this job is CHF 0.22
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 1.25        │ 0.25        │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 1.45        │ 0.25        │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)
Submitted batch job 54909048


In [28]:
from pathlib import Path

import numpy as np


def parse_score_file(score_file):
    """Parse MaSIF comma-separated score summary (see process_search_outputs.py)."""
    score_file = Path(score_file)
    if not score_file.is_file():
        return None
    with score_file.open() as f:
        return {part.split(": ")[0]: part.split(": ")[1] for part in f.read().split(", ")}


def gather_search_results(results_root, query_target):
    """Collect .score fields and 4x4 transforms under results_root/query_target/site_*."""
    target_dir = Path(results_root) / query_target
    if not target_dir.is_dir():
        raise FileNotFoundError(f"No search output directory: {target_dir}")

    rows = []
    for site_dir in sorted(target_dir.glob("site_*")):
        target_info = parse_score_file(site_dir / "target_info.txt")
        target_vix = int(target_info["vix"]) if target_info and "vix" in target_info else None

        for score_path in sorted(site_dir.glob("*/*.score")):
            summary = parse_score_file(score_path)
            if summary is None:
                continue

            transform_path = score_path.with_name(f"{score_path.stem}_transform.npy")
            if not transform_path.is_file():
                continue

            transform = np.load(transform_path)
            if transform.shape != (4, 4):
                print(f"Skipping invalid transform shape {transform.shape}: {transform_path}")
                continue

            rows.append(
                {
                    "target": query_target,
                    "target_site": site_dir.name,
                    "target_vix": target_vix,
                    "matched_protein": summary["name"],
                    "matched_patch_id": int(summary["point id"]),
                    "score": float(summary["score"]),
                    "desc_dist_score": float(summary["desc_dist_score"]),
                    "clashing_ca": int(summary["clashing_ca"]),
                    "clashing_heavy": int(summary["clashing_heavy"]),
                    "matched_vix": int(summary["match_vix"])
                    if "match_vix" in summary
                    else None,
                    "desc_dist": float(summary["desc_dist"])
                    if "desc_dist" in summary
                    else None,
                    "iface_score": float(summary["iface_score"])
                    if "iface_score" in summary
                    else None,
                    "mean_desc_dist_score": float(summary["mean_desc_dist_score"])
                    if "mean_desc_dist_score" in summary
                    else None,
                    "flattened_transform": ",".join(map(str, transform.flatten())),
                }
            )

    return pd.DataFrame(rows)


results_csv = os.path.join(query_out_dir, f"{QUERY_TARGET}_search_results.csv")
df_results = gather_search_results(query_out_dir, QUERY_TARGET)
df_results.to_csv(results_csv, index=False)
print(f"Wrote {len(df_results)} matches to {results_csv}")
df_results.head()


Wrote 5 matches to /scratch/ymeng/NNeosurf/masif-neosurf/data/masif_search/8VLB_A/8VLB_A_search_results.csv


,target,target_site,target_vix,matched_protein,matched_patch_id,score,desc_dist_score,clashing_ca,clashing_heavy,matched_vix,desc_dist,iface_score,mean_desc_dist_score,flattened_transform
0,8VLB_A,site_10,1151,5QSO_A,0,0.9207,20.208859,0,3,2970,1.860577,0.368014,0.132084,"0.1891158143360388,-0.5745362750716273,0.79633..."
1,8VLB_A,site_22,876,7UWO_AB,0,0.9724,10.627931,0,0,3781,1.983913,0.418113,0.100263,"0.8372511317894467,-0.3670006845399405,0.40536..."
2,8VLB_A,site_32,3051,5QSV_D,13,0.9433,20.194905,0,2,5576,1.900894,0.559067,0.160277,"0.8865497883281478,0.40543432257998213,-0.2228..."
3,8VLB_A,site_32,3051,5QSV_D,5,0.9466,20.453185,0,4,2937,1.946821,0.617374,0.159791,"0.8761196444031263,0.31449496918257425,-0.3653..."
4,8VLB_A,site_4,440,6H0F_B,7,0.9111,23.760661,0,0,4092,1.965906,0.866648,0.204833,"0.6092110780241586,-0.4468527467828541,0.65512..."
